# Chapter 6: Self-Attention (Single Head)

[Read this chapter online](https://jackluu.io/book/section-2-attention/ch06-self-attention/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch06-self-attention.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 6: Self-Attention (Single Head)

![Where we are in the big picture](../assets/diagrams/ch06-where-we-are.png){ width="756" }
*Figure 6.1: Attention builds context by looking across the sequence.*

Now that our tokens have meaning through embeddings (as shown in Chapter 5), reading one word at a time is not enough. To understand "it" in "the trophy didn't fit in the suitcase because it was too big", you have to connect "it" back to "trophy". In this chapter you will:

- Learn how tokens "look" at each other to build context.
- Use the Query, Key, Value (QKV) system to find relevant information.
- See how attention is just a weighted average of numbers.
- Apply a causal mask to hide the future.

**Words to Know**
    - **Self-Attention**: A mechanism where tokens evaluate every other token in the sequence to gather context.
    - **Query (Q)**: A vector representing what a token is looking for.
    - **Key (K)**: A vector representing what a token contains.
    - **Value (V)**: A vector holding the actual information to be shared.
    - **Causal Mask**: A filter that prevents tokens from seeing future tokens.
    
## Theory

### The Problem: Context Matters

If a model only considers one token at a time, it has no memory. It cannot connect subjects to verbs, or adjectives to nouns. Every token needs to "look at" the other tokens and decide which ones matter most to its own meaning. 

### The QKV System

Self-attention solves this using three vectors for every token:

- **Query (Q)**: What am I looking for?
- **Key (K)**: What do I offer?
- **Value (V)**: What information do I actually contain?

Imagine you are at a networking event. You are looking for a marketing expert (your Query). Someone is wearing a badge that says "Marketing Director" (their Key). When your Query matches their Key, you start a conversation and absorb their advice (their Value).

In our model, every token creates its own Q, K, and V vectors by multiplying its embedding by learned weights. A token acts as a seeker (Query) and a source (Key and Value) at the exact same time.

### Attention is a Weighted Average

![Attention is a weighted average](../assets/diagrams/ch06-weighted-average.png){ width="458" }
*Figure 6.2: Token 5 blends information from previous tokens.*

Here is the "aha" moment: attention is nothing more than a weighted average (Figure 6.2). When token 5 calculates its final value, it does not just pick the single best token to look at. It mixes them all. 

First, we multiply all Queries by all Keys (`q @ k.transpose`) to get an attention score for every pair. We scale these down by dividing by `sqrt(head_size)` (the size of each attention head, which is 32 here) so the numbers do not get too large. Then we apply `softmax` to turn these scores into percentages (weights) that sum to 1.0 (or 100%).

For example, when token 5 looks at the sequence, it might assign weights like this (as plotted in Figure 6.3). These values are rounded for display, so they add up to about 100% (100.1%):

- Token 0: 19.9%
- Token 1: 20.4%
- Token 2: 12.2%
- Token 3: 12.1%
- Token 4: 21.4%
- Token 5: 14.1%

![Attention weights for token 5](../assets/diagrams/ch06-attention-weights.png){ width="650" }
*Figure 6.3: A plotted bar chart of the attention weights.*

Token 5's final output is simply 19.9% of the value of token 0, plus 20.4% of the value of token 1, and so on for all six tokens. The "Numbers Machine" builds context by blending the numbers of the most relevant past tokens. This context gathering completes the "Attention" stage of our map (Figure 6.1).

### The Causal Mask

![The causal mask hides the future](../assets/diagrams/ch06-causal-mask.png){ width="235" }
*Figure 6.4: A lower triangular mask ensures tokens only see the past.*

There is a catch: the model processes all tokens at once. If we let token 5 look at token 6, it would be cheating. During training, it would just copy the answer from the future instead of learning to predict it. 

We apply a **causal mask** (a triangle of negative infinities, shown in Figure 6.4) to the scores before the softmax step. Softmax turns `-infinity` into exactly `0.0`. This ensures that token 5 pays 0% attention to tokens 6, 7, 8, and 9. It can only see itself and the past.

**In Business**
    When parsing a customer support chat, a simple keyword search treats every word independently. A context-aware system uses self-attention. It connects the word "refund" in message 4 back to the "broken screen" mentioned in message 1, understanding the full conversation history before it generates a reply in the company's house style.

## Code

![How the code flows in self-attention](../assets/diagrams/ch06-code-flow.png){ width="458" }
*Figure 6.5: The Q, K, V vectors are created, scored, masked, and then multiplied.*

We use `src/ch05_self_attention.py` to build the `SingleHeadAttention` class (flow illustrated in Figure 6.5).

```python
q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        # Compute attention scores and scale them to keep variance stable
        scale = HEAD_SIZE ** -0.5
        scores = q @ k.transpose(-2, -1) * scale

        # Mask future tokens by setting their scores to negative infinity
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        # Convert scores to probabilities and apply dropout
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        # Compute final output by taking weighted sum of values
        out = weights @ v
```

Here is what happens when we run it:

```python
$ python src/ch05_self_attention.py
--- Attention weights (what token 5 attends to) ---
Token 5 attends to tokens 0..5 (future tokens masked):
  token 0: 0.199  #####
  token 1: 0.204  ######
  token 2: 0.122  ###
  token 3: 0.121  ###
  token 4: 0.214  ######
  token 5: 0.141  ####
  token 6: 0.000
  token 7: 0.000
  token 8: 0.000
  token 9: 0.000

Self-attention done! Ready for Chapter 7.
```

**What just happened:**

- Lines 1 to 3 created independent Query, Key, and Value vectors.
- Line 7 computed the attention scores between all tokens.
- Line 10 masked the future tokens (notice tokens 6 to 9 have exactly 0.000 weight).
- Line 17 created the output (for example, token 5 created its output by taking a weighted average of tokens 0 through 5).

**Shape Check:** Table 6.1 lists the shapes of the attention variables.

**Table 6.1:** Tensor shapes during the self-attention calculation.

| Variable | Shape | Meaning |
|---|---|---|
| `q`, `k`, `v` | `[B, T, head_size]` | Queries, Keys, and Values. `head_size` is 32. |
| `scores` | `[B, T, T]` | Attention scores between every pair of tokens. |
| `weights` | `[B, T, T]` | The softmax probabilities (summing to 1 per row). |
| `out` | `[B, T, head_size]` | The final context-aware output vectors. |

## Try It

**Try It**
    Remove the `scale` division in `src/ch05_self_attention.py` by changing it to `scale = 1.0`. Run it again. Notice how the weights become much more extreme (some very close to 1.0, others 0.0). The scaling is critical to keep the model flexible and learning smoothly.

**Watch Out**
    Be careful with the `transpose` step. We only want to swap the last two dimensions (Time and `head_size`) to compute the dot product properly. If you use `.T`, it might flip the Batch dimension too, crashing your shape calculations. Always use `.transpose(-2, -1)`.

## Key Takeaways

- Self-attention builds context by letting tokens "look" at other tokens.
- The QKV system works like a search engine: Queries match with Keys to retrieve Values.
- Attention is just a weighted average of the Values.
- A causal mask sets future scores to negative infinity, preventing the model from cheating.

## Check Your Understanding

1. What is the difference between a Query and a Key?
2. Why do we divide the attention scores by the square root of the head size?
3. What happens to a score of negative infinity when passed through softmax?
4. Why is the causal mask necessary for a language model?


## Further Reading

**The first attention.** Translation models of the day read the whole source sentence, squeezed it into a single fixed vector, and wrote the translation from that. Long sentences did not survive the squeeze. The fix: let the model look back over every input word and decide, at each output word, which ones matter right now. That weighted look-back is attention. The Transformer three years later kept this idea and threw out everything around it.

**The architecture this book builds.** Reading a sequence one step at a time is slow, because step 500 cannot start until step 499 has finished, and distant words stay hard to connect. This paper removed the step-by-step reading entirely and kept only attention, plus a note of each token's position. Every token can then be processed at once, which is what made training on very large amounts of text practical. The model you build in Chapters 6 to 10 is this design, made small.

<div class="refs" markdown>

Bahdanau, D., Cho, K., & Bengio, Y. (2014). *Neural machine translation by jointly learning to align and translate* (arXiv:1409.0473). arXiv. https://doi.org/10.48550/arXiv.1409.0473

Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, L., & Polosukhin, I. (2017). *Attention is all you need* (arXiv:1706.03762). arXiv. https://doi.org/10.48550/arXiv.1706.03762

</div>

---

### `src/ch05_self_attention.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch05_self_attention.py"   # a cell has none, and the file uses it to find the text

"""
Implement single-head self-attention.
This file belongs to Chapter 6.
Run: python src/ch05_self_attention.py
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

import os
import sys

from src.utils.config import GPTConfig

# Settings
config = GPTConfig()
HEAD_SIZE = config.n_embd // config.n_heads

# --- The Idea ---
class SingleHeadAttention(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        C = config.n_embd

        # Linear layers to compute query, key, and value vectors
        self.query = nn.Linear(C, head_size, bias=False)
        self.key   = nn.Linear(C, head_size, bias=False)
        self.value = nn.Linear(C, head_size, bias=False)

        # Lower triangular matrix to prevent attending to future tokens
        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(config.block_size, config.block_size)
            )
        )
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        # Compute attention scores and scale them to keep variance stable
        scale = self.head_size ** -0.5
        scores = q @ k.transpose(-2, -1) * scale

        # Mask future tokens by setting their scores to negative infinity
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))

        # Convert scores to probabilities and apply dropout
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        # Compute final output by taking weighted sum of values
        out = weights @ v
        return out

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 6: Self-Attention (Single Head)\n")
    print(f"Config: n_embd={config.n_embd}, head_size={HEAD_SIZE}")

    head = SingleHeadAttention(HEAD_SIZE)
    total_params = sum(p.numel() for p in head.parameters())
    print(f"SingleHeadAttention parameters: {total_params:,}")

    B, T = 2, 10
    x = torch.randn(B, T, config.n_embd)
    out = head(x)
    print(f"\nInput shape : {x.shape}")
    print(f"Output shape: {out.shape}   (B, T, head_size)")

    print("\n--- Attention weights (what token 5 attends to) ---")
    # Disable gradient tracking since we're just analyzing weights
    with torch.no_grad():
        q = head.query(x)
        k = head.key(x)
        scale = HEAD_SIZE ** -0.5
        scores = q @ k.transpose(-2, -1) * scale
        scores = scores.masked_fill(head.tril[:T, :T] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)

    w = weights[0, 5, :].tolist()
    print("Token 5 attends to tokens 0..5 (future tokens masked):")
    for i, wi in enumerate(w):
        bar = "#" * int(wi * 30)
        print(f"  token {i}: {wi:.3f}  {bar}")

    print("\nSelf-attention done! Ready for Chapter 7.")